# VisionGuard AI — Phase 3.4: YOLOv8 Nano Training & Evaluation

**Project:** VisionGuard AI — Infrastructure Road Damage Inspection Platform  
**Phase:** Phase 3.4 — Model Training, Validation, and Held-Out Test Evaluation  
**Target Model:** YOLOv8 Nano (`yolov8n.pt`)  
**Dataset:** RDD2022 India Subset (7,706 images: 5,356 train, 1,150 val, 1,200 test)  
**Target Classes:** `0: D00_longitudinal_crack`, `1: D10_transverse_crack`, `2: D20_alligator_crack`, `3: D40_pothole`  
**Execution Platform:** Google Colab (Cloud GPU Runtime)  

---

### Instructions for Running in Google Colab
1. **Select GPU Runtime:** Go to **Runtime** > **Change runtime type**, select **T4 GPU** under Hardware accelerator, and click **Save**.
2. **Verify Dataset:** If starting a fresh session, Step 2 will automatically restore your prepared dataset (`visionguard_dataset`) from your Google Drive backup.
3. **Execute Sequentially:** Run cells from top to bottom. Do not skip validation cells.
4. **Quarantine Test Split:** The held-out test split is strictly evaluated in Step 9 **only after** training and validation checkpoint selection are finalized.

## Step 1: Runtime & Hardware Verification

Verify GPU model, available VRAM, CUDA capability, and ephemeral disk space in the current Google Colab environment.

In [ ]:
# 1. Check GPU hardware and VRAM allocation
import os
import shutil
import subprocess
import torch

print("=== Google Colab GPU & System Diagnostics ===")
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"Active GPU Device:      {gpu_name}")
    print(f"Total Dedicated VRAM:   {total_vram_gb} GB")
    if total_vram_gb < 10.0:
        print("WARNING: VRAM is below 10 GB. If training runs Out-Of-Memory, reduce batch size to 4 or 8.")
else:
    print("\n[CRITICAL WARNING] No GPU detected! Please configure Colab:")
    print("  -> Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

total_b, used_b, free_b = shutil.disk_usage('/content')
print(f"Available Disk Space:   {free_b / (1024**3):.2f} GB free in /content")

## Step 2: Dependency Verification & Ultralytics Installation

Install and verify the official `ultralytics` package and essential computer vision libraries.

In [ ]:
# 2. Install and verify Ultralytics and dependencies
%pip install -q --upgrade ultralytics opencv-python pillow matplotlib pyyaml pandas

import sys
import yaml
import cv2
import PIL
import matplotlib
import ultralytics
from ultralytics import YOLO

print("=== Environment & Library Baseline ===")
print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"OpenCV Version:      {cv2.__version__}")
print(f"Pillow Version:      {PIL.__version__}")
print(f"Matplotlib Version:  {matplotlib.__version__}")
print("[PASSED] Dependencies verified.")

## Step 3: Dataset Restoration, Integrity & Split Verification

Verify that the prepared dataset exists at `/content/visionguard_dataset`.  
If starting from a new Colab session, restore the dataset from the Google Drive backup archive (`visionguard_rdd2022_india_yolo.zip`).

In [ ]:
# 3. Locate or restore prepared dataset and verify data.yaml
import os
import zipfile
from google.colab import drive

DATASET_DIR = '/content/visionguard_dataset'
YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

# Check if dataset is already extracted
if not os.path.exists(YAML_PATH):
    print("Dataset not found in local /content. Attempting to restore from Google Drive...")
    try:
        drive.mount('/content/drive')
        drive_backup = '/content/drive/MyDrive/VisionGuard_Artifacts/visionguard_rdd2022_india_yolo.zip'
        local_backup = '/content/visionguard_rdd2022_india_yolo.zip'
        
        if os.path.exists(drive_backup):
            print(f"Found backup in Drive: {drive_backup}. Restoring...")
            with zipfile.ZipFile(drive_backup, 'r') as zf:
                zf.extractall('/content')
        elif os.path.exists(local_backup):
            print(f"Found local zip: {local_backup}. Restoring...")
            with zipfile.ZipFile(local_backup, 'r') as zf:
                zf.extractall('/content')
        else:
            raise FileNotFoundError("Dataset backup zip not found in Drive or /content! Please run Phase 3.3 preparation first.")
    except Exception as e:
        print(f"Dataset restoration failed: {e}")
        raise

assert os.path.exists(YAML_PATH), f"Missing dataset configuration file at {YAML_PATH}!"

with open(YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

print("=== Dataset Configuration (data.yaml) ===")
print(yaml.dump(data_config, default_flow_style=False))

# Verify split counts and disjointness
split_images = {}
split_labels = {}

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    assert os.path.isdir(img_dir), f"Directory not found: {img_dir}"
    assert os.path.isdir(lbl_dir), f"Directory not found: {lbl_dir}"
    
    imgs = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    lbls = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.lower().endswith('.txt'))
    
    split_images[split] = imgs
    split_labels[split] = lbls
    print(f"Split '{split}': {len(imgs):,} images | {len(lbls):,} label files")

# Safety assertions
assert len(split_images['train']) > 0, "Train set is empty!"
assert len(split_images['val']) > 0, "Val set is empty!"
assert len(split_images['test']) > 0, "Test set is empty!"

train_val_overlap = split_images['train'] & split_images['val']
train_test_overlap = split_images['train'] & split_images['test']
val_test_overlap = split_images['val'] & split_images['test']

assert len(train_val_overlap) == 0, f"Data leakage detected between train and val: {len(train_val_overlap)} files!"
assert len(train_test_overlap) == 0, f"Data leakage detected between train and test: {len(train_test_overlap)} files!"
assert len(val_test_overlap) == 0, f"Data leakage detected between val and test: {len(val_test_overlap)} files!"

print("\n[PASSED] Split integrity verified: All 3 splits are strictly disjoint.")

## Step 4: Class Imbalance Audit (Pre-Training Distribution)

Audit instance tallies across the training split to establish baseline expectations for each damage category, with special attention to `D10` (transverse cracks) scarcity.

In [ ]:
# 4. Detailed class distribution audit across train split
from collections import defaultdict

train_lbl_dir = os.path.join(DATASET_DIR, 'labels', 'train')
class_counts = defaultdict(int)
image_counts = defaultdict(int)
background_images = 0

for lbl_file in os.listdir(train_lbl_dir):
    if not lbl_file.endswith('.txt'): continue
    path = os.path.join(train_lbl_dir, lbl_file)
    with open(path, 'r') as f:
        lines = f.readlines()
    
    if len(lines) == 0:
        background_images += 1
    else:
        seen_in_img = set()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                cid = int(parts[0])
                class_counts[cid] += 1
                seen_in_img.add(cid)
        for cid in seen_in_img:
            image_counts[cid] += 1

class_names = data_config['names']
total_objects = sum(class_counts.values())

print("=== Training Split Distress Distribution Audit ===")
print(f"Total Training Images:     {len(os.listdir(train_lbl_dir)):,}")
print(f"Images with Damage:       {len(os.listdir(train_lbl_dir)) - background_images:,}")
print(f"Sound / Background Images: {background_images:,}")
print(f"Total Bounding Boxes:     {total_objects:,}\n")

print(f"{'Class ID':<10} | {'Class Name':<25} | {'Instances':<12} | {'% Share':<10} | {'Images':<10}")
print("-" * 75)
for cid in sorted(class_names.keys()):
    cname = class_names[cid]
    count = class_counts[cid]
    pct = (count / total_objects * 100) if total_objects > 0 else 0
    n_imgs = image_counts[cid]
    print(f"{cid:<10} | {cname:<25} | {count:<12,} | {pct:>6.2f}%    | {n_imgs:<10,}")
print("-" * 75)

# Explicit notice on class imbalance
print("\n[CRITICAL NOTE ON CLASS IMBALANCE]:")
print(" - Potholes (D40) and Alligator cracks (D20) comprise ~76% of all training instances.")
print(" - Transverse cracks (D10) comprise <1.5% of instances. Its recall will be naturally constrained.")
print(" - We will evaluate each class independently to avoid masking D10 performance behind overall mAP.")

## Step 5: Model Initialization & Configuration

Initialize YOLOv8 Nano (`yolov8n.pt`) with pretrained weights for transfer learning, configure hyperparameter settings (`imgsz=640`, `batch=8`, `epochs=30`, `seed=42`, `patience=10`), and prepare experiment logging directories.

In [ ]:
# 5. Initialize model and set hyperparameters
from ultralytics import YOLO

EXPERIMENT_PROJECT = '/content/visionguard_runs'
EXPERIMENT_NAME = 'yolov8n_rdd_india_exp1'

os.makedirs(EXPERIMENT_PROJECT, exist_ok=True)

print("Initializing YOLOv8 Nano baseline...")
model = YOLO('yolov8n.pt')

# Documented Training Configuration
TRAINING_CONFIG = {
    'data': YAML_PATH,
    'epochs': 30,             # Training budget
    'imgsz': 640,             # Standard resolution
    'batch': 8,               # Memory-safe batch size for Colab T4
    'patience': 10,           # Early stopping patience
    'seed': 42,               # Exact reproducibility
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 2,
    'project': EXPERIMENT_PROJECT,
    'name': EXPERIMENT_NAME,
    'exist_ok': True,
    'verbose': True,
    'plots': True,
    'save': True,
    'val': True               # Validate every epoch for checkpoint selection
}

print("\n=== Planned Training Hyperparameters ===")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k:<12}: {v}")

## Step 6: Training Execution

Execute training on the training split while monitoring loss convergence and validation metrics. The validation set is used for periodic evaluation and best-checkpoint selection (`best.pt`).

In [ ]:
# 6. Execute YOLOv8 Nano training
import time

start_time = time.time()
print(f"Starting YOLOv8 Nano training at {time.strftime('%Y-%m-%d %H:%M:%S')}...")
print("This process typically takes 30–50 minutes on a Tesla T4 GPU.\n")

train_results = model.train(**TRAINING_CONFIG)

elapsed_min = (time.time() - start_time) / 60.0
print(f"\n[COMPLETED] Training finished in {elapsed_min:.2f} minutes.")

RUN_DIR = os.path.join(EXPERIMENT_PROJECT, EXPERIMENT_NAME)
BEST_WEIGHTS_PATH = os.path.join(RUN_DIR, 'weights', 'best.pt')
LAST_WEIGHTS_PATH = os.path.join(RUN_DIR, 'weights', 'last.pt')

assert os.path.exists(BEST_WEIGHTS_PATH), f"Checkpoint not found at {BEST_WEIGHTS_PATH}!"
print(f"Best model weights saved at: {BEST_WEIGHTS_PATH} ({round(os.path.getsize(BEST_WEIGHTS_PATH) / 1e6, 2)} MB)")

## Step 7: Validation Performance & Loss Curve Diagnostics

Inspect the generated training and validation loss curves, mAP progression, and confusion matrix from the completed training run.

In [ ]:
# 7. Render training curves and validation diagnostics
import matplotlib.pyplot as plt
from PIL import Image

results_png = os.path.join(RUN_DIR, 'results.png')
confusion_matrix_png = os.path.join(RUN_DIR, 'confusion_matrix.png')

if os.path.exists(results_png):
    plt.figure(figsize=(16, 10))
    plt.imshow(Image.open(results_png))
    plt.title("YOLOv8 Training and Validation Loss Curves", fontsize=14)
    plt.axis('off')
    plt.show()
else:
    print("results.png not found.")

if os.path.exists(confusion_matrix_png):
    plt.figure(figsize=(10, 10))
    plt.imshow(Image.open(confusion_matrix_png))
    plt.title("Validation Confusion Matrix", fontsize=14)
    plt.axis('off')
    plt.show()
else:
    print("confusion_matrix.png not found.")

## Step 8: Validation Metrics & Class-Wise Performance Analysis

Run formal validation of `best.pt` on the validation set, extracting precision, recall, mAP@0.5, and mAP@0.5:0.95 across each individual damage class.

In [ ]:
# 8. Formal validation evaluation of best checkpoint
best_model = YOLO(BEST_WEIGHTS_PATH)

print("Running validation evaluation on validation split...")
val_metrics = best_model.val(
    data=YAML_PATH,
    split='val',
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.6,
    project=EXPERIMENT_PROJECT,
    name='validation_eval',
    exist_ok=True
)

print("\n" + "="*75)
print("           VALIDATION SPLIT EVALUATION REPORT (best.pt)")
print("="*75)
print(f"Mean Precision:     {val_metrics.box.mp:.4f}")
print(f"Mean Recall:        {val_metrics.box.mr:.4f}")
print(f"mAP@0.5:            {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:       {val_metrics.box.map:.4f}")
print("-"*75)
print(f"{'Class ID':<10} | {'Class Name':<25} | {'Precision':<12} | {'Recall':<10} | {'mAP@0.5':<10}")
print("-"*75)

for i, cid in enumerate(sorted(class_names.keys())):
    cname = class_names[cid]
    p = val_metrics.box.p[i] if i < len(val_metrics.box.p) else 0.0
    r = val_metrics.box.r[i] if i < len(val_metrics.box.r) else 0.0
    ap50 = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0.0
    print(f"{cid:<10} | {cname:<25} | {p:<12.4f} | {r:<10.4f} | {ap50:<10.4f}")
print("="*75)

## Step 9: Final Evaluation on the Held-Out Test Set

> [!IMPORTANT]
> **Strict Protocol:** The held-out test set (`1,200 images`) is evaluated **only now**, after model weights and hyperparameters are frozen. It was never seen during training or checkpoint selection.

In [ ]:
# 9. Formal evaluation on held-out test split
print("Running final evaluation on quarantined held-out TEST split...")
test_metrics = best_model.val(
    data=YAML_PATH,
    split='test',
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.6,
    project=EXPERIMENT_PROJECT,
    name='test_eval',
    exist_ok=True
)

print("\n" + "="*75)
print("           HELD-OUT TEST SET EVALUATION REPORT (best.pt)")
print("="*75)
print(f"Mean Precision:     {test_metrics.box.mp:.4f}")
print(f"Mean Recall:        {test_metrics.box.mr:.4f}")
print(f"mAP@0.5:            {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:       {test_metrics.box.map:.4f}")
print("-"*75)
print(f"{'Class ID':<10} | {'Class Name':<25} | {'Precision':<12} | {'Recall':<10} | {'mAP@0.5':<10}")
print("-"*75)

for i, cid in enumerate(sorted(class_names.keys())):
    cname = class_names[cid]
    p = test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0.0
    r = test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0.0
    ap50 = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0.0
    print(f"{cid:<10} | {cname:<25} | {p:<12.4f} | {r:<10.4f} | {ap50:<10.4f}")
print("="*75)

## Step 10: Qualitative Predictions Visualization on Test Images

Run inference on sample test images and render predicted bounding boxes, defect labels, and detection confidence scores.

In [ ]:
# 10. Visualize predictions on sample test images
test_img_dir = os.path.join(DATASET_DIR, 'images', 'test')
sample_test_files = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png'))][:8]

plt.figure(figsize=(16, 8))
for i, s_file in enumerate(sample_test_files):
    img_path = os.path.join(test_img_dir, s_file)
    preds = best_model.predict(source=img_path, conf=0.25, verbose=False)[0]
    annotated_img = preds.plot()
    annotated_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(annotated_rgb)
    plt.title(s_file, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Step 11: Model Checkpoint Export & Google Drive Backup

Save the best checkpoint (`best.pt`), export evaluation summaries to JSON, and back up the weights to Google Drive.

In [ ]:
# 11. Save model weights and summary to Google Drive
import json

DRIVE_ARTIFACTS_DIR = '/content/drive/MyDrive/VisionGuard_Artifacts'
os.makedirs(DRIVE_ARTIFACTS_DIR, exist_ok=True)

# Export summary JSON
summary = {
    'model': 'YOLOv8 Nano (yolov8n.pt)',
    'dataset': 'RDD2022 India Subset',
    'epochs_trained': 30,
    'image_size': 640,
    'batch_size': 8,
    'validation_metrics': {
        'mean_precision': float(val_metrics.box.mp),
        'mean_recall': float(val_metrics.box.mr),
        'mAP_50': float(val_metrics.box.map50),
        'mAP_50_95': float(val_metrics.box.map)
    },
    'test_metrics': {
        'mean_precision': float(test_metrics.box.mp),
        'mean_recall': float(test_metrics.box.mr),
        'mAP_50': float(test_metrics.box.map50),
        'mAP_50_95': float(test_metrics.box.map)
    },
    'classes': data_config['names']
}

summary_json_path = os.path.join(RUN_DIR, 'experiment_summary.json')
with open(summary_json_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Copy best.pt to Drive
drive_best_pt = os.path.join(DRIVE_ARTIFACTS_DIR, 'best.pt')
drive_summary = os.path.join(DRIVE_ARTIFACTS_DIR, 'experiment_summary.json')

shutil.copyfile(BEST_WEIGHTS_PATH, drive_best_pt)
shutil.copyfile(summary_json_path, drive_summary)

print(f"[SUCCESS] Best weights backed up to Drive: {drive_best_pt} ({round(os.path.getsize(drive_best_pt) / 1e6, 2)} MB)")
print(f"[SUCCESS] Summary backed up to Drive:      {drive_summary}")

print("\n--- HOW TO DOWNLOAD TO LOCAL LAPTOP FOR PHASE 4 FLASK INTEGRATION ---")
print("Option A: Open Google Drive in your browser, navigate to 'VisionGuard_Artifacts', and download 'best.pt'.")
print("Option B: Run in Colab: from google.colab import files; files.download(BEST_WEIGHTS_PATH)")
print("Place the downloaded 'best.pt' file into your local project at: visionGuard/ai/models/best.pt")

## Step 12: Phase 3.4 Completion & Engineering Guardrails Notice

> [!CAUTION]
> **Engineering Guardrails:**  
> 1. A 2D bounding box localizes distress instances and calculates pixel dimensions, but **does not measure true physical crack width** without camera calibration and ground-plane homography.  
> 2. Statistical model confidence measures detection certainty, **not structural safety or severity**.  
> 3. Transverse crack (`D10`) metrics reflect real-world data scarcity in the Indian road network.